In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import shutil
import re

## 시퀀스의 앞부분(CLS 근처) 부분에 스코어가 높게 나타나는 경향을 경향을 보정하기 위한 작업 진행

In [ ]:
#d = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/analysis_raw_data/VAT score/VATP_raw/DBP_original_VATP.csv')
#nd = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/analysis_raw_data/VAT score/VATP_raw/NDBP_original_VATP.csv')
#dbp_pad_only_AA = pd.concat([d,nd],axis=1)
dbp_pad_only_AA = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/analysis_raw_data/VAT score/VATP_raw/RBP_VATP_nopad_only_AA_251030.csv')
dbp_pad_only_AA

In [ ]:
import matplotlib.pyplot as plt

protein_id = "ENSP00000002125"  # 보고 싶은 단백질 컬럼 이름

plt.figure(figsize=(10,4))
plt.plot(dbp_pad_only_AA[protein_id].values, color='royalblue', linewidth=1)
plt.title(f"Z-score profile across sequence ({protein_id})", fontsize=13)
plt.xlabel("Amino acid position", fontsize=12)
plt.ylabel("Z-normalized score", fontsize=12)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.show()

## 생물학적으로 앞 부분에 도메인이 전체적으로 균등할 확률 (10%) 기대치보다 높은지 확인 작업 먼저 진행

In [ ]:
interpro_consensus = pd.read_csv('/Users/hanjin/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/interpro_consensus/human_interpro_consensus.csv')
interpro_consensus

In [ ]:
IPR_coverage = pd.read_csv('/Users/hanjin/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/interpro_consensus/IPR_coverage_in_protein.csv')
IPR_coverage

In [ ]:
seq_len = IPR_coverage[['ENSP_ID','seq_len']]
seq_len

In [ ]:
interpro_consensus = interpro_consensus.merge(seq_len, on='ENSP_ID',how='left')
interpro_consensus

In [ ]:
# 0~1 정규화
interpro_consensus['norm_start'] = interpro_consensus['consensus_start'] / interpro_consensus['seq_len']
interpro_consensus['norm_end']   = interpro_consensus['consensus_end']   / interpro_consensus['seq_len']

# head(0.0~0.1), body(0.1~1.0)와의 겹침 길이(정규화 길이)
interpro_consensus['overlap_head'] = (
    np.clip(np.minimum(interpro_consensus['norm_end'], 0.1) - 
            np.maximum(interpro_consensus['norm_start'], 0.0), 0, None)
)
interpro_consensus['overlap_body'] = (
    np.clip(np.minimum(interpro_consensus['norm_end'], 1.0) - 
            np.maximum(interpro_consensus['norm_start'], 0.1), 0, None)
)

# 이분법적 겹침 여부 (있으면 True)
interpro_consensus['is_head_domain'] = interpro_consensus['overlap_head'] > 0
interpro_consensus['is_body_domain'] = interpro_consensus['overlap_body'] > 0

# 단백질 단위 비율
protein_has_head = interpro_consensus.groupby('ENSP_ID')['is_head_domain'].any()
protein_has_body = interpro_consensus.groupby('ENSP_ID')['is_body_domain'].any()

print(f"N-terminal (0–10%) domain ratio: {protein_has_head.mean():.2%}")
print(f"Body (10–100%) domain ratio: {protein_has_body.mean():.2%}")

# (선택) 길이 가중 coverage (잔기 비율)로 단백질별 head/body IPR 밀도도 계산
g = interpro_consensus.groupby('ENSP_ID')
head_cov = (g.apply(lambda x: (x['overlap_head'] * x['seq_len']).sum()) / g['seq_len'].first()).rename('ipr_head_cov')
body_cov = (g.apply(lambda x: (x['overlap_body'] * x['seq_len']).sum()) / g['seq_len'].first()).rename('ipr_body_cov')
ipr_cov  = pd.concat([head_cov, body_cov], axis=1)

## 이후 z-score 정규화 및 head 부분이 나머지 부분보다 높은 단백질에 대한 보정 진행할지 여부 파악

In [ ]:
# VATP: 각 열이 단백질(ENSP_ID), 각 행이 residue position
# 예시: VATP = pd.read_csv('DBP_model_VATP_251030.csv', index_col=0)

normalized_df = dbp_pad_only_AA.copy()

for col in normalized_df.columns:
    mu = normalized_df[col].mean(skipna=True)
    sigma = normalized_df[col].std(skipna=True)
    
    if sigma == 0 or pd.isna(sigma):
        # 분산이 0이거나 결측이면 0으로 채움
        normalized_df[col] = 0
    else:
        normalized_df[col] = (normalized_df[col] - mu) / sigma
normalized_df

In [ ]:
DBP_postive_pred = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/DBP_with_means_positive_pred.csv')
DBP_postive_pred_list = DBP_postive_pred.drop_duplicates(subset='ENSP_ID')['ENSP_ID'].tolist()
len(DBP_postive_pred_list)

In [ ]:
normalized_df_pos = normalized_df[DBP_postive_pred_list]
normalized_df_neg = normalized_df.drop(columns=DBP_postive_pred_list)
normalized_df_neg

In [ ]:
normalized_df.to_csv('/Users/khj/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/analysis_raw_data/VAT score/VATP_raw/DBP_VATP_nopad_only_AA_zscore_251031.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import os

# 그래프 저장 디렉토리

for protein_id in normalized_df.columns:
    plt.figure(figsize=(10, 4))
    plt.plot(normalized_df[protein_id].values, color='royalblue', linewidth=1)
    plt.title(f"Z-score profile across sequence ({protein_id})", fontsize=13)
    plt.xlabel("Amino acid position", fontsize=12)
    plt.ylabel("Z-normalized score", fontsize=12)
    plt.axhline(0, color='gray', linestyle='--', linewidth=1)
    
    plt.tight_layout()
    plt.show() 
    plt.close()  # 메모리 절약

In [ ]:
normalized_df.describe().T[['mean', 'std']].head()

In [ ]:
import numpy as np
import pandas as pd

def make_bias_summary(normalized_df: pd.DataFrame, head_frac: float = 0.10) -> pd.DataFrame:
    """
    normalized_df: 열=ENSP_ID, 행=AA 위치의 z-score VAT (단백질마다 유효 길이는 NaN으로 구분되어도 OK)
    head_frac    : head로 보는 상대 길이 비율 (기본 10%)
    반환: index=ENSP_ID, columns=[seq_len, head_len, body_len, head_mean, body_mean, delta]
    """
    records = []
    for ens in normalized_df.columns:
        s = normalized_df[ens].dropna()             # 단백질 유효 구간만 사용
        L = int(len(s))
        if L < 3:
            continue
        H = max(1, int(np.floor(L * head_frac)))    # head 길이(최소 1)
        head = s.iloc[:H]
        body = s.iloc[H:]

        if len(body) == 0:  # 전부 head인 극단 케이스 방어
            continue

        head_mean = float(head.mean())
        body_mean = float(body.mean())
        delta = head_mean - body_mean

        records.append({
            "ENSP_ID": ens,
            "seq_len": L,
            "head_len": int(len(head)),
            "body_len": int(len(body)),
            "head_mean": head_mean,
            "body_mean": body_mean,
            "delta": delta
        })

    bias_summary = pd.DataFrame.from_records(records).set_index("ENSP_ID").sort_values("delta", ascending=False)
    return bias_summary

In [ ]:

# 사용 예시
# bias_summary: 각 단백질의 head/body 평균과 Δ(head−body)
for i in np.arange(0, 1, 0.1):
    bias_summary = make_bias_summary(normalized_df, head_frac=i)

    # 빠른 요약 확인
    print({
        "head_frac": i,
        "N_proteins": bias_summary.shape[0],
        "delta_median": float(bias_summary["delta"].median()),
        "head_positive_ratio": float((bias_summary["delta"] > 0).mean()),
})

In [ ]:
# PAD version {'N_proteins': 18221, 'delta_median': 1.9208085189202648, 'head_positive_ratio': 0.9999451182701279}
# NO PAD version {'N_proteins': 18221, 'delta_median': 1.8450793587785541, 'head_positive_ratio': 1.0}
# Original version {'N_proteins': 18221, 'delta_median': 0.851378160810278, 'head_positive_ratio': 0.5678612589868832}
# only postive prediction {'N_proteins': 2410, 'delta_median': 1.7956139958938215, 'head_positive_ratio': 1.0}
# only negative prediction {'N_proteins': 15811, 'delta_median': 1.8569132953171712, 'head_positive_ratio': 1.0}

In [ ]:
normalized_df

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import UnivariateSpline

# =========================================================
# 0) wide -> long (+ 상대 위치)
# ---------------------------------------------------------
def to_long_with_pos(df_wide: pd.DataFrame):
    """
    df_wide: rows = residue index (0..maxL-1), cols = proteins, values = raw VAT (length 이후 NaN)
    return:
      df_long: ['protein','idx','pos','score','L']
      Ls:     protein별 길이 Series
    """
    # 각 단백질 길이(유효 값 수)
    Ls = df_wide.notna().sum(axis=0).astype(int)

    # (idx, protein) -> score, NaN 제외
    s = df_wide.stack(dropna=True)             # MultiIndex
    idx  = s.index.get_level_values(0).to_numpy()
    prot = s.index.get_level_values(1).to_numpy()
    score = s.to_numpy()

    # 길이 매핑
    L_map = Ls.reindex(pd.Index(prot)).to_numpy()
    # 상대 위치 (0~1), 중심 정렬
    pos = (idx + 0.5) / np.maximum(L_map, 1)

    df_long = pd.DataFrame({
        "protein": prot,
        "idx": idx.astype(int),
        "pos": pos.astype(float),
        "score": score.astype(float),
        "L": L_map.astype(int)
    })
    return df_long, Ls


# =========================================================
# 1) 전역 위치 스플라인 f(pos), g(pos) 적합 (raw 기준)
# ---------------------------------------------------------
def fit_positional_spline_from_wide(
    df_wide: pd.DataFrame,
    nbins: int = 300,
    per_protein_equal_weight: bool = True,
    fit_std: bool = True,
    spline_s: float = 0.002,
    k: int = 3,
):
    """
    raw VAT로 전역 위치 평균 f(pos) 및 (선택) 표준편차 g(pos) 스플라인을 적합
    - per_protein_equal_weight=True: 단백질별 기여 균등화 (권장)
    반환:
      f_func(pos_array) -> np.ndarray
      g_func(pos_array) -> np.ndarray or None
      diag = (bin_mid, bin_mean, bin_std, bin_counts)
    """
    df_long, _ = to_long_with_pos(df_wide)
    bins = np.linspace(0.0, 1.0, nbins + 1, dtype=float)
    bin_mid = 0.5 * (bins[1:] + bins[:-1])

    if per_protein_equal_weight:
        # 단백질-빈 평균을 먼저 구해 각 단백질의 영향력을 균등화
        df_long["bin"] = np.clip(np.digitize(df_long["pos"].to_numpy(), bins) - 1, 0, nbins - 1)
        grp = (
            df_long
            .groupby(["protein", "bin"], observed=True)["score"]
            .mean()
            .reset_index()
        )
        bin_stats = grp.groupby("bin", observed=True)["score"].agg(["mean", "std", "count"])
        bin_counts = grp.groupby("bin", observed=True)["protein"].nunique()
        # 인덱스 보정/채움
        bin_stats  = bin_stats.reindex(range(nbins))
        bin_counts = bin_counts.reindex(range(nbins)).fillna(0)

        bin_mean = bin_stats["mean"].to_numpy()
        bin_std  = bin_stats["std"].to_numpy()
        bin_counts = bin_counts.to_numpy(dtype=float)

    else:
        # residue 수 기준 가중(긴 단백질의 기여가 커짐)
        idx = np.clip(np.digitize(df_long["pos"].to_numpy(), bins) - 1, 0, nbins - 1)
        vals = df_long["score"].to_numpy()

        sums   = np.bincount(idx, weights=vals, minlength=nbins).astype(float)
        counts = np.bincount(idx, minlength=nbins).astype(float)

        with np.errstate(invalid="ignore", divide="ignore"):
            bin_mean = np.where(counts > 0, sums / counts, np.nan)

        sq_sums = np.bincount(idx, weights=vals**2, minlength=nbins).astype(float)
        var = np.where(counts > 0, (sq_sums / counts) - bin_mean**2, np.nan)
        var[var < 0] = 0.0
        bin_std = np.sqrt(var)
        bin_counts = counts

    # 유효 bin 선택
    mask = (bin_counts > 0) & np.isfinite(bin_mean)
    u = bin_mid[mask]
    m = bin_mean[mask]
    w = bin_counts[mask].astype(float)

    # f(pos) 스플라인 적합 (가중치 = 유효 표본수)
    f_spline = UnivariateSpline(u, m, w=w, s=spline_s * max(len(u), 1), k=k)

    def f_func(x):
        x = np.asarray(x, dtype=float)
        return f_spline(x)

    g_func = None
    if fit_std:
        s_std = bin_std[mask]
        # std NaN 안전치 대체(중앙값)
        safe_std = np.where(np.isfinite(s_std), s_std, np.nanmedian(s_std))
        g_spline = UnivariateSpline(u, safe_std, w=w, s=spline_s * max(len(u), 1), k=k)

        def g_func(x):
            x = np.asarray(x, dtype=float)
            return g_spline(x)

    diag = (bin_mid, bin_mean, bin_std, bin_counts)
    return f_func, g_func, diag


# =========================================================
# 2) 같은 raw에 보정 적용
# ---------------------------------------------------------
def apply_positional_correction_to_wide(
    df_wide: pd.DataFrame,
    f_func,
    g_func=None,
    eps: float = 1e-6
):
    """
    raw VAT df_wide에 대해:
      resid = raw - f(pos)
      zcorr = resid / (g(pos)+eps)  (g_func 제공 시)
    반환:
      df_resid (wide), df_z (wide or None)
    """
    df_long, _ = to_long_with_pos(df_wide)

    f = f_func(df_long["pos"].to_numpy())
    resid = df_long["score"].to_numpy() - f

    df_resid = pd.DataFrame(index=df_wide.index, columns=df_wide.columns, dtype=float)
    tmp = pd.DataFrame({"protein": df_long["protein"], "idx": df_long["idx"], "resid": resid})
    for prot, sub in tmp.groupby("protein", sort=False):
        df_resid.loc[sub["idx"].to_numpy(), prot] = sub["resid"].to_numpy()

    if g_func is None:
        return df_resid, None

    g = g_func(df_long["pos"].to_numpy()) + eps
    # g_func 생성 직후/또는 apply 단계에서
    g = g_func(df_long["pos"].to_numpy())
    floor = np.nanpercentile(g, 5) * 0.5  # 예: 하위 5%의 절반
    g = np.maximum(g, max(floor, 1e-6))
    zcorr = resid / (g + 1e-12)

    df_z = pd.DataFrame(index=df_wide.index, columns=df_wide.columns, dtype=float)
    tmpz = pd.DataFrame({"protein": df_long["protein"], "idx": df_long["idx"], "zcorr": zcorr})
    for prot, sub in tmpz.groupby("protein", sort=False):
        df_z.loc[sub["idx"].to_numpy(), prot] = sub["zcorr"].to_numpy()

    return df_resid, df_z


# =========================================================
# 3) 단일 조건 요약: head bias / positional profile
# ---------------------------------------------------------
def summarize_single_condition(
    df_wide: pd.DataFrame,
    start_frac: float = 0.10,
    pad_len: int = 0,
    head_mode: str = "max",   # 'frac'|'abs'|'max'
):
    """
    한 벌의 VAT(예: raw 또는 보정값)에 대해,
    앞구간(head)과 나머지(body)의 평균 차이를 단백질별로 계산하고 집단 요약을 반환
    """
    rows = []
    for col in df_wide.columns:
        s = df_wide[col].dropna().astype(float)
        L = len(s)
        if L < 3:
            rows.append({"protein": col, "head_bias": np.nan,
                         "front_mean": np.nan, "body_mean": np.nan, "L": L})
            continue

        if head_mode == "frac":
            head_n = max(1, int(L * start_frac))
        elif head_mode == "abs":
            head_n = min(L, max(1, int(pad_len)))
        elif head_mode == "max":
            head_n = min(L, max(max(1, int(L * start_frac)), int(pad_len)))
        else:
            raise ValueError("head_mode must be one of {'frac','abs','max'}")

        front_mean = float(np.mean(s.iloc[:head_n]))
        body_mean  = float(np.mean(s.iloc[head_n:])) if L > head_n else np.nan
        head_bias  = front_mean - body_mean if np.isfinite(body_mean) else np.nan

        rows.append({
            "protein": col,
            "head_bias": head_bias,
            "front_mean": front_mean,
            "body_mean": body_mean,
            "L": L,
            "head_n": int(head_n),
        })

    df = pd.DataFrame(rows).set_index("protein")
    summary = {
        "N_proteins": int(df.shape[0]),
        "head_bias_median": float(df["head_bias"].median(skipna=True)),
        "front_mean_median": float(df["front_mean"].median(skipna=True)),
        "body_mean_median": float(df["body_mean"].median(skipna=True)),
    }
    return summary, df


def positional_profile_single(
    df_wide: pd.DataFrame,
    nbins: int = 300,
    per_protein_equal_weight: bool = True
):
    """
    한 벌의 VAT(예: raw 또는 보정값)에 대해,
    상대 위치 0~1을 nbins로 나눠 bin 평균 곡선을 반환 (전역 위치 프로파일)
    """
    long, _ = to_long_with_pos(df_wide)
    bins = np.linspace(0.0, 1.0, nbins + 1, dtype=float)
    long["bin"] = np.clip(np.digitize(long["pos"].to_numpy(), bins) - 1, 0, nbins - 1)

    if per_protein_equal_weight:
        g = long.groupby(["protein", "bin"], observed=True)["score"].mean().reset_index()
        prof = g.groupby("bin", observed=True)["score"].mean().reindex(range(nbins))
    else:
        prof = long.groupby("bin", observed=True)["score"].mean().reindex(range(nbins))

    return pd.DataFrame({
        "bin_mid": 0.5 * (bins[1:] + bins[:-1]),
        "mean_score": prof.to_numpy()
    })

In [ ]:
# 0) 입력: VAT_wide_raw  (행=pos, 열=protein, 길이 이후 NaN), raw 점수
VAT_wide_raw = dbp_pad_only_AA.copy()

# 1) 전역 위치 스플라인 적합 (raw 기준)
f_func, g_func, diag = fit_positional_spline_from_wide(
    df_wide=VAT_wide_raw,
    nbins=300,
    per_protein_equal_weight=True,  # 단백질별 기여 균등화
    fit_std=True,                   # 위치별 표준편차까지 추정 → zcorr 사용 가능
    spline_s=0.002,                 # 0.001~0.01 사이에서 데이터에 맞게 튜닝
    k=3
)

# 2) 같은 raw에 보정 적용 → resid / zcorr (둘 다 wide 형태, NaN 동일 유지)
VAT_resid, VAT_zcorr = apply_positional_correction_to_wide(
    df_wide=VAT_wide_raw,
    f_func=f_func,
    g_func=g_func,      # 분산보정 원치 않으면 None
    eps=1e-6
)

# 3) 단일 조건 요약: 보정 전/후 head bias 비교
pre_sum,  pre_detail  = summarize_single_condition(
    VAT_wide_raw, start_frac=0.10, pad_len=0, head_mode="max"
)
post_sum, post_detail = summarize_single_condition(
    VAT_resid,     start_frac=0.10, pad_len=0, head_mode="max"
)
postz_sum, postz_detail = summarize_single_condition(
    VAT_zcorr,     start_frac=0.10, pad_len=0, head_mode="max"
) if VAT_zcorr is not None else (None, None)

print("[RAW]   ", pre_sum)    # 보정 전 전역 head bias 요약
print("[RESID] ", post_sum)   # f(pos) 제거 후 요약
print("[ZCORR] ", postz_sum)  # 분산까지 보정한 요약(선택)

# 4) 전역 위치 프로파일(0~1) 비교용 곡선(시각화는 원하는 라이브러리로)
prof_raw = positional_profile_single(VAT_wide_raw, nbins=300, per_protein_equal_weight=True)
prof_res = positional_profile_single(VAT_resid,     nbins=300, per_protein_equal_weight=True)
prof_z   = positional_profile_single(VAT_zcorr,     nbins=300, per_protein_equal_weight=True) if VAT_zcorr is not None else None

In [ ]:
x = np.linspace(0, 1, 300)
y = f_func(x)
plt.plot(x, y)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# 1) 전역 위치 프로파일: 보정 전/후 평탄도 비교
# ---------------------------------------------------------
plt.figure(figsize=(7,4))
plt.plot(prof_raw["bin_mid"], prof_raw["mean_score"], label="Raw (original)", linewidth=2)
plt.plot(prof_res["bin_mid"], prof_res["mean_score"], label="Resid (after f(pos) correction)", linewidth=2)
if prof_z is not None:
    plt.plot(prof_z["bin_mid"], prof_z["mean_score"], label="Zcorr (f,g corrected)", linewidth=2, linestyle="--")
plt.axhline(0, linestyle="--", color="gray", linewidth=1)
plt.xlabel("Relative position (0 → 1)")
plt.ylabel("Mean VAT score")
plt.title("Global positional VAT profile (before / after correction)")
plt.legend()
plt.tight_layout()
plt.show()


# -----------------------------
# 2) 단백질별 head_bias 분포 (boxplot)
# -----------------------------
frames = [
    pre_detail[["head_bias"]].assign(which="Raw"),
    post_detail[["head_bias"]].assign(which="Resid")
]
if postz_detail is not None:
    frames.append(postz_detail[["head_bias"]].assign(which="Zcorr"))

# ✅ 인덱스 중복 방지
plot_df = pd.concat(frames, axis=0, ignore_index=True)
# 또는: plot_df = pd.concat(frames, axis=0, ignore_index=False).reset_index(names="protein")

# NaN 제거 (그룹에 NaN만 있으면 seaborn이 또 불평할 수 있어요)
plot_df = plot_df.dropna(subset=["head_bias"])

plt.figure(figsize=(6,4))
sns.boxplot(data=plot_df, x="which", y="head_bias", palette="Set2", showfliers=False)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.ylabel("Head bias (front_mean − body_mean)")
plt.title("Per-protein head bias distribution")
plt.tight_layout()
plt.show()


# -----------------------------
# 3) 앞/뒤 평균 비교 (scatter)
# -----------------------------
def _prep(df, label):
    d = df[["front_mean", "body_mean"]].copy()
    d = d.dropna()
    d["which"] = label
    return d

scatter_df = pd.concat([
    _prep(pre_detail, "Raw"),
    _prep(post_detail, "Resid"),
    _prep(postz_detail, "Zcorr") if postz_detail is not None else pd.DataFrame(columns=["front_mean","body_mean","which"])
], ignore_index=True)

plt.figure(figsize=(5,5))
for label, sub in scatter_df.groupby("which"):
    plt.scatter(sub["body_mean"], sub["front_mean"], s=10, alpha=0.5, label=label)

# 축 한계는 데이터 기반으로
x_min = float(scatter_df["body_mean"].min()) if len(scatter_df) else -1
x_max = float(scatter_df["body_mean"].max()) if len(scatter_df) else 1
y_min = float(scatter_df["front_mean"].min()) if len(scatter_df) else -1
y_max = float(scatter_df["front_mean"].max()) if len(scatter_df) else 1
lims_min = min(x_min, y_min)
lims_max = max(x_max, y_max)
plt.plot([lims_min, lims_max], [lims_min, lims_max], 'k--', alpha=0.6)  # y=x
plt.xlim(lims_min, lims_max)
plt.ylim(lims_min, lims_max)

plt.xlabel("Body mean")
plt.ylabel("Front mean")
plt.title("Front vs Body mean per protein")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def normalize_per_protein(df_wide: pd.DataFrame, robust=False):
    df_norm = pd.DataFrame(index=df_wide.index, columns=df_wide.columns, dtype=float)
    for col in df_wide.columns:
        s = df_wide[col].dropna().astype(float)
        if len(s) < 3:
            continue
        if robust:
            med = np.median(s)
            mad = np.median(np.abs(s - med)) * 1.4826
            normed = (s - med) / (mad if mad > 1e-8 else 1e-8)
        else:
            mu, sd = np.mean(s), np.std(s, ddof=0)
            normed = (s - mu) / (sd if sd > 1e-8 else 1e-8)
        df_norm.loc[s.index, col] = normed
    return df_norm

In [ ]:
VAT_zcorr_normed = normalize_per_protein(VAT_zcorr, robust=True)
VAT_zcorr_normed

In [ ]:
stats = VAT_zcorr_normed.stack(dropna=True).describe()
print(stats)

In [ ]:
protein_id = "ENSP00000393538"  # 보고 싶은 단백질 컬럼 이름

plt.figure(figsize=(10,4))
plt.plot(dbp_pad_only_AA[protein_id].values, color='royalblue', linewidth=1)
plt.title(f"Z-score profile across sequence ({protein_id})", fontsize=13)
plt.xlabel("Amino acid position", fontsize=12)
plt.ylabel("Z-normalized score", fontsize=12)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(VAT_zcorr_normed[protein_id].values, color='royalblue', linewidth=1)
plt.title(f"Z-score profile across sequence ({protein_id})", fontsize=13)
plt.xlabel("Amino acid position", fontsize=12)
plt.ylabel("Z-normalized score", fontsize=12)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.show()

In [ ]:
pd.set_option('display.max_rows', None)

work = 'D'
search = protein_id

search = search.replace('>','')

interproscan = pd.read_csv(f'/Users/khj/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/analysis_raw_data/interproscan/Interproscan_result_250704.tsv',sep='\t',header=None)

filtered_df = interproscan[interproscan.apply(lambda row: row.astype(str).str.contains(search, case=False).any(), axis=1)]

sort_column = filtered_df.columns[6]

# 해당 열 기준으로 오름차순 정렬
filtered_df = filtered_df.sort_values(by=sort_column)

selected = filtered_df.iloc[:, [0, 3, 5, 6, 7, 11, 12,13]]
selected.columns = ['Accession','DB','Description','Start','End','Interpro ID', 'Information','GO']
selected = selected[['Accession','DB','Start','End','Interpro ID','Information','Description','GO']]

selected

In [ ]:
VAT_zcorr_normed.to_csv('/Users/khj/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/analysis_raw_data/VAT score/spline_z_score_normalized/RBP_spline_zscore_normalized_251103.csv', index=False)